In [ ]:
import pandas as pd
import re
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
import spacy
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold

Read the dataset

In [ ]:
nlp = spacy.load("en_core_web_sm")

news_df = pd.read_csv('/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv')
df = news_df
# news_df.head()

This section of code is used for preprocessing and data cleaning, collected from another notebook of ours.

In [ ]:
# Handling missing values
news_df.fillna('', inplace=True)

# Removing duplicate rows
news_df = news_df.drop_duplicates()

# Converting to lowercase
news_df['title'] = news_df['title'].str.lower()
news_df['text'] = news_df['text'].str.lower()

# Removing punctuation
news_df['title'] = news_df['title'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
news_df['text'] = news_df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# Removing titles with less than 3 word, texts with less than 50 words
news_df = news_df[news_df['title'].apply(lambda x: len(x.split()) >= 3)]
news_df = news_df[news_df['text'].apply(lambda x: len(x.split()) >= 50)]

In [ ]:
def tokenize_and_lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

news_df['titleTokens'] = news_df['title'].apply(tokenize_and_lemmatize)
news_df['textTokens'] = news_df['text'].apply(tokenize_and_lemmatize)

Create a bag of words dictionary for each class and in total. For creating the file, we will go through each "document" and add  all of our data to the bag of words dictionary. We will ignore all of the words which will have only special characters (dot, comma, paranthesis etc.), and also numbers due to not actually helping a lot in terms of sentiment analysis.

The article of the dataset says that the data is split in 6 key pair categories which are relevant: AUDUSD, EURCHF, EURUSD, GBPUSD, and USDJPY. We will apply naive bayes with this variable in addition as a prior knowledge, with the assumption that it is independent with the data and classes. "t" will be the pair class.

$$P(c|d,t) ∝ P(d,t|c)*P(c) = P(d|c)*P(t|c)*P(c) $$



This function will help us create a dictionary "Bag of Words" from scratch, to help us calculate by their occurences of each word in each class

In [ ]:
def create_bag_of_words_occurences_ticker(df, label):
  bag_dict = {
      "Positive": {},
      "Positive_count": 0,
      "Positive_count_frequency": 0,
      "Neutral": {},
      "Neutral_count": 0,
      "Neutral_count_frequency": 0,
      "Negative": {},
      "Negative_count": 0,
      "Negative_count_frequency": 0,
      "total": {},
      "total_count_frequency": 0,
      "total_count": len(df)
  }

  regex_filter = "[A-Za-z]+"

  for i in df.index:

    bag_dict[df["true_sentiment"][i] + '_count'] += 1
    doc_tokenized = [el.lower() for el in df[label + 'Tokens'][i] if re.search(regex_filter, el)]
    occurence_words = {}
    ticker = df["ticker"][i]
    if ticker not in bag_dict[df["true_sentiment"][i]]:
      bag_dict[df["true_sentiment"][i]][ticker] = 1
    else:
      bag_dict[df["true_sentiment"][i]][ticker] += 1

    if ticker not in bag_dict["total"]:
      bag_dict["total"][ticker] = 1
    else:
      bag_dict["total"][ticker] += 1

    for el in doc_tokenized:
      if el not in occurence_words:
        occurence_words[el] = 0
        if el not in bag_dict[df["true_sentiment"][i]]:
          bag_dict[df["true_sentiment"][i]][el] = 1
        else:
          bag_dict[df["true_sentiment"][i]][el] += 1
        if el not in bag_dict["total"]:
          bag_dict["total"][el] = 1
        else:
          bag_dict["total"][el] += 1
        bag_dict[df["true_sentiment"][i] + '_count_frequency'] += 1
        bag_dict["total_count_frequency"] += 1

  return bag_dict

This function will be used to get the predictions from another dataframe (test part). Also, for the prediction, LaPlace smoothing is added, due to words which are not present in the bag of words generated.

In [ ]:
def get_predictions_ticker(bag_of_words_dict, df, label):
  prediction = []
  regex_filter = "[A-Za-z]+"
  classes = ["Positive", "Neutral", "Negative"]
  for i in df.index:

      doc_tokenized = [el.lower() for el in df[label + 'Tokens'][i] if re.search(regex_filter, el)]

      results = {}
      V_len = len(bag_of_words_dict["total"].keys())
      for cls in classes:
        results[cls] = np.log(bag_of_words_dict[cls + "_count"] / bag_of_words_dict["total_count"])
        results[cls] += np.log(bag_of_words_dict[cls][df["ticker"][i]] / bag_of_words_dict["total"][df["ticker"][i]])
        cls_freq = bag_of_words_dict[cls + "_count_frequency"]
        for el in doc_tokenized:
          # we apply LaPlace (adding + 1 to each word)
          word_count = bag_of_words_dict[cls][el] + 1 if el in bag_of_words_dict[cls] else 1
          results[cls] += np.log(word_count / (cls_freq + V_len))

      prediction.append(max(results, key=results.get))

  return prediction

As now we have our bag of words generator and we can generate prediction for each class, we will go to the next step.

In this section of code, we will apply Naive Bayes with a train/test split of 80/20, and evaluate the predictions with the ground truth.

In [ ]:
df_train, df_test = train_test_split(news_df, test_size=0.2, random_state=42)

label = "title"
print("Occurences metrics for the title:")
bag_of_words_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker(bag_of_words_dict, df_test, label)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Occurences metrics for the title:
              precision    recall  f1-score   support

    Positive       0.69      0.74      0.71       155
     Neutral       0.66      0.59      0.62       161
    Negative       0.66      0.68      0.67       139

    accuracy                           0.67       455
   macro avg       0.67      0.67      0.67       455
weighted avg       0.67      0.67      0.67       455



Applying KFolds to the Naive Bayes

In [ ]:
kf = KFold(5)#cut in 5 for test and train
label = "title"

for i, (train_index, test_index) in enumerate(kf.split(news_df)):
  df_train = news_df.iloc[train_index]
  df_test = news_df.iloc[test_index]
  print("Kfold {}".format(i))
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker(text_dict, df_test, label)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold 0
              precision    recall  f1-score   support

    Positive       0.57      0.66      0.61       134
     Neutral       0.54      0.49      0.51       147
    Negative       0.64      0.62      0.63       174

    accuracy                           0.59       455
   macro avg       0.59      0.59      0.59       455
weighted avg       0.59      0.59      0.59       455

Kfold 1
              precision    recall  f1-score   support

    Positive       0.54      0.70      0.61       137
     Neutral       0.61      0.57      0.59       155
    Negative       0.73      0.60      0.66       163

    accuracy                           0.62       455
   macro avg       0.63      0.62      0.62       455
weighted avg       0.63      0.62      0.62       455

Kfold 2
              precision    recall  f1-score   support

    Positive       0.70      0.64      0.67       170
     Neutral       0.69      0.54      0.60       170
    Negative       0.50      0.74      0.60       1

In [ ]:
!pip freeze > '/content/drive/MyDrive/HLT project/Final code/requirements_Your_Model.txt'